In [1]:
import re
from pathlib import Path
import pandas as pd
from openpyxl import load_workbook

In [ ]:
FILE_PATH = 

TOTAL_MARKER = "Всего по БВУ:"


BANK_COL = 2  
DATA_START_COL_DEFAULT = 3  


In [3]:
def is_date_like(v) -> bool:
    """Определяем, похоже ли значение на дату (datetime или строка формата dd.mm.yyyy / dd-mm-yyyy и т.п.)."""
    if v is None:
        return False
    if hasattr(v, "year") and hasattr(v, "month") and hasattr(v, "day"):
        return True
    if isinstance(v, str):
        s = v.strip()
        # простая проверка на 01.04.2007 / 1.4.2007 / 01-04-2007
        return bool(re.match(r"^\d{1,2}[.\-/]\d{1,2}[.\-/]\d{2,4}$", s))
    return False


def to_date(v):
    """Приводим к pandas.Timestamp (или None)."""
    if v is None:
        return None
    if hasattr(v, "year") and hasattr(v, "month") and hasattr(v, "day"):
        return pd.to_datetime(v).normalize()
    if isinstance(v, str):
        s = v.strip().replace("-", ".").replace("/", ".")
        try:
            return pd.to_datetime(s, dayfirst=True).normalize()
        except Exception:
            return None
    return None


def clean_text(v):
    if v is None:
        return None
    s = str(v).strip()
    s = re.sub(r"\s+", " ", s)
    return s if s else None


def looks_like_bank_name(s: str) -> bool:
    """Грубая эвристика: строка не пустая, не TOTAL_MARKER и не 'без ...' заголовок и т.п."""
    if not s:
        return False
    if TOTAL_MARKER.lower() in s.lower():
        return False
    # часто встречается 'без БТА' как подзаголовок — это не банк
    if s.lower().startswith("без "):
        return False
    # если это похоже на "Активы ...", "Требования ..." — тоже заголовок
    if any(k in s.lower() for k in ["содержание"]): #"актив", "требован", "срок", "итого", 
        return False
    return True


In [4]:
def find_date_row(ws, start_row, bank_col=BANK_COL, scan_up=3, scan_down=3):
    """
    Находим строку, где расположены даты для блока.
    Обычно это рядом со строкой TOTAL_MARKER: чуть выше/ниже.
    Возвращает (date_row_idx, first_date_col, last_date_col).
    """
    candidates = list(range(max(1, start_row - scan_up), start_row + scan_down + 1))
    best = None

    for r in candidates:
        # ищем подряд несколько date-like значений начиная с DATA_START_COL_DEFAULT
        date_cols = []
        for c in range(DATA_START_COL_DEFAULT, ws.max_column + 1):
            v = ws.cell(r, c).value
            if is_date_like(v):
                date_cols.append(c)
            else:
                # даты обычно идут плотным блоком, можно прервать после "окна"
                # но чтобы не обрывать слишком рано, допускаем 2 пустых подряд
                pass

        # считаем "плотный" интервал дат
        if len(date_cols) >= 3:
            first_c = min(date_cols)
            last_c = max(date_cols)
            best = (r, first_c, last_c)
            break

    return best


def parse_sheet(ws, sheet_name: str):
    """
    Парсим один лист.
    Ищем все блоки, где есть строка 'Всего по БВУ' (может быть в A, B, ...).
    Для каждого блока:
      - bank_col определяется по позиции 'Всего по БВУ'
      - measure = значение строго на 1 строку выше в bank_col
      - даты ищем рядом (как раньше), но старт дат = max(DATA_START_COL_DEFAULT, bank_col+1)
      - конец блока: 3 пустых строки подряд в bank_col
    """
    rows_out = []

    # 1) Найдём все позиции TOTAL_MARKER (row, col) в верхней левой части листа
    # (обычно маркер в первых нескольких колонках)
    total_pos = []
    SEARCH_COLS = 8  # можно увеличить, если нужно
    for r in range(1, ws.max_row + 1):
        found = None
        for c in range(1, SEARCH_COLS + 1):
            v = clean_text(ws.cell(r, c).value)
            if v and TOTAL_MARKER.lower() in v.lower():
                found = (r, c)
                break
        if found:
            total_pos.append(found)

    # 2) Для каждого блока
    for idx, (total_r, bank_col_block) in enumerate(total_pos):
        next_total_r = total_pos[idx + 1][0] if idx + 1 < len(total_pos) else ws.max_row + 1

        # measure — строго текст из ячейки над "Всего по БВУ" (в той же колонке, что и маркер)
        measure = clean_text(ws.cell(total_r - 1, bank_col_block).value) if total_r > 1 else None
        if not measure:
            measure = f"Measure_from_{ws.title}_row_{total_r}"

        # ---- поиск строки дат рядом с total_r, но учитываем что даты идут после bank_col_block ----
        def find_date_row_local(start_row, scan_up=3, scan_down=3):
            candidates = list(range(max(1, start_row - scan_up), start_row + scan_down + 1))
            for rr in candidates:
                date_cols = []
                start_c = bank_col_block + 1
                for cc in range(start_c, ws.max_column + 1):
                    if is_date_like(ws.cell(rr, cc).value):
                        date_cols.append(cc)
                if len(date_cols) >= 3:
                    return rr, min(date_cols), max(date_cols)
            return None

        date_info = find_date_row_local(total_r)
        if not date_info:
            continue

        date_row, first_c, last_c = date_info

        dates = []
        cols = []
        for c in range(first_c, last_c + 1):
            d = to_date(ws.cell(date_row, c).value)
            if d is not None:
                dates.append(d)
                cols.append(c)

        if not dates:
            continue

        # данные в блоке: от total_r и вниз, пока не 3 пустых подряд в bank_col_block
        start_data_r = total_r
        end_data_r = next_total_r - 1

        empty_streak = 0
        for r in range(start_data_r, end_data_r + 1):
            bank_name = clean_text(ws.cell(r, bank_col_block).value)

            if bank_name is None:
                empty_streak += 1
                if empty_streak >= 3:
                    break
                continue
            else:
                empty_streak = 0

            # пропускаем подзаголовки типа "без БТА" (если нужно)
            if (TOTAL_MARKER.lower() not in bank_name.lower()) and (not looks_like_bank_name(bank_name)):
                continue

            for d, c in zip(dates, cols):
                val = ws.cell(r, c).value
                if val is None or val == "":
                    continue

                if isinstance(val, (int, float)):
                    amount = float(val)
                else:
                    s = str(val).replace(" ", "").replace("\u00A0", "").replace(",", ".")
                    try:
                        amount = float(s)
                    except Exception:
                        continue

                rows_out.append(
                    {
                        "Date": d,
                        "Bank_name": bank_name,
                        "Measure": measure,
                        "Amount": amount,
                        "Sheet": ws.title,  # имя вкладки Excel
                    }
                )

    return pd.DataFrame(rows_out)



In [5]:
wb = load_workbook(FILE_PATH, data_only=True)

all_dfs = []
for sheet_name in wb.sheetnames:
    ws = wb[sheet_name]
    df_sheet = parse_sheet(ws, sheet_name)
    if not df_sheet.empty:
        all_dfs.append(df_sheet)

result = pd.concat(all_dfs, ignore_index=True) if all_dfs else pd.DataFrame(columns=["Date","Bank_name","Measure","Amount","Sheet"])

result.shape, result.head()


((447569, 5),
         Date      Bank_name Measure       Amount   Sheet
 0 2000-05-31  Всего по БВУ:  Активы  355258270.0  Активы
 1 2000-06-30  Всего по БВУ:  Активы  391736573.0  Активы
 2 2000-07-31  Всего по БВУ:  Активы  410405364.0  Активы
 3 2000-08-31  Всего по БВУ:  Активы  430510139.0  Активы
 4 2000-10-31  Всего по БВУ:  Активы  478472161.0  Активы)

In [ ]:
from pathlib import Path

FILES = [

]



In [ ]:
from openpyxl import load_workbook
import pandas as pd

def parse_workbook(file_path: str) -> pd.DataFrame:
    wb = load_workbook(file_path, data_only=True)
    dfs = []
    for sheet_name in wb.sheetnames:
        ws = wb[sheet_name]
        df_sheet = parse_sheet(ws, sheet_name) 
        if not df_sheet.empty:
            dfs.append(df_sheet)

    out = pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame(
        columns=["Date","Bank_name","Measure","Amount","Sheet"]
    )
    out["SourceFile"] = Path(file_path).name
    return out


In [8]:
all_results = []
for fp in FILES:
    df_one = parse_workbook(fp)
    all_results.append(df_one)

result = pd.concat(all_results, ignore_index=True) if all_results else pd.DataFrame()

result.shape, result.head()


C:\ProgramData\anaconda3\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
C:\ProgramData\anaconda3\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


((1167646, 6),
         Date      Bank_name Measure       Amount   Sheet       SourceFile
 0 2000-05-31  Всего по БВУ:  Активы  355258270.0  Активы  Assets_OLD.xlsx
 1 2000-06-30  Всего по БВУ:  Активы  391736573.0  Активы  Assets_OLD.xlsx
 2 2000-07-31  Всего по БВУ:  Активы  410405364.0  Активы  Assets_OLD.xlsx
 3 2000-08-31  Всего по БВУ:  Активы  430510139.0  Активы  Assets_OLD.xlsx
 4 2000-10-31  Всего по БВУ:  Активы  478472161.0  Активы  Assets_OLD.xlsx)

In [ ]:
result["Date"] = pd.to_datetime(result["Date"], errors="coerce")
result["Amount"] = pd.to_numeric(result["Amount"], errors="coerce")
result = result.dropna(subset=["Date","Bank_name","Measure","Amount"])

out_path = Path(FILES[0]).parent / "melted_all_files_assets.parquet"
result.to_parquet(out_path, index=False)

out_path

In [ ]:
result[(result['Date']=='2007-04-01')&(result['Measure']=='Активы в иностранной валюте до года, тыс. тенге')]

In [ ]:
from pathlib import Path

FILES = [
 
]




In [ ]:
from openpyxl import load_workbook
import pandas as pd

def parse_workbook(file_path: str) -> pd.DataFrame:
    wb = load_workbook(file_path, data_only=True)
    dfs = []
    for sheet_name in wb.sheetnames:
        ws = wb[sheet_name]
        df_sheet = parse_sheet(ws, sheet_name)  
        if not df_sheet.empty:
            dfs.append(df_sheet)

    out = pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame(
        columns=["Date","Bank_name","Measure","Amount","Sheet"]
    )
    out["SourceFile"] = Path(file_path).name
    return out


In [ ]:
all_results = []
for fp in FILES:
    df_one = parse_workbook(fp)
    all_results.append(df_one)

result = pd.concat(all_results, ignore_index=True) if all_results else pd.DataFrame()

result.shape, result.head()


In [ ]:
result["Date"] = pd.to_datetime(result["Date"], errors="coerce")
result["Amount"] = pd.to_numeric(result["Amount"], errors="coerce")
result = result.dropna(subset=["Date","Bank_name","Measure","Amount"])

out_path = Path(FILES[0]).parent / "melted_all_files_liabilities.parquet"
result.to_parquet(out_path, index=False)

out_path


In [ ]:
result

In [ ]:
from pathlib import Path

FILES = [

]



In [ ]:
from openpyxl import load_workbook
import pandas as pd

def parse_workbook(file_path: str) -> pd.DataFrame:
    wb = load_workbook(file_path, data_only=True)
    dfs = []
    for sheet_name in wb.sheetnames:
        ws = wb[sheet_name]
        df_sheet = parse_sheet(ws, sheet_name)  
        if not df_sheet.empty:
            dfs.append(df_sheet)

    out = pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame(
        columns=["Date","Bank_name","Measure","Amount","Sheet"]
    )
    out["SourceFile"] = Path(file_path).name
    return out


In [ ]:
all_results = []
for fp in FILES:
    df_one = parse_workbook(fp)
    all_results.append(df_one)

result = pd.concat(all_results, ignore_index=True) if all_results else pd.DataFrame()

result.shape, result.head()


In [ ]:
result["Date"] = pd.to_datetime(result["Date"], errors="coerce")
result["Amount"] = pd.to_numeric(result["Amount"], errors="coerce")
result = result.dropna(subset=["Date","Bank_name","Measure","Amount"])

out_path = Path(FILES[0]).parent / "melted_all_files_ie.parquet"
result.to_parquet(out_path, index=False)

out_path


In [ ]:
result

In [ ]:
result['SourceFile'].unique()